In [ ]:

import os, sys, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0=time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
base=os.path.dirname(glob.glob("/kaggle/input/**/items_human.parquet", recursive=True)[0])
prev=os.path.dirname(glob.glob("/kaggle/input/**/features_human.npy", recursive=True)[0])
os.makedirs("/kaggle/working/src",exist_ok=True)
code_dir=os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code_dir+"/*.py"): shutil.copy(p,"/kaggle/working/src/")
open("/kaggle/working/src/__init__.py","a").close()
os.makedirs("/kaggle/working/models",exist_ok=True)
shutil.copy(code_dir+"/anti_words.json","/kaggle/working/models/anti_words.json")
os.chdir("/kaggle/working"); sys.path.insert(0,"/kaggle/working")
from sklearn.feature_extraction.text import TfidfVectorizer
from src.attr_features import parse, compare, FEATURE_NAMES
from src.name_features import parse_name, compare_names, build_idf, NAME_FEATURE_NAMES
from src.string_features import compare_strings, STRING_FEATURE_NAMES
from src.neighbour_features import build as nb_build, compare as nb_compare, NEIGHBOUR_FEATURE_NAMES
from src.brand_features import colours, canonical, compare_brands, compare_colours, BRAND_FEATURE_NAMES
from src.dim_features import dimensions, compare_dimensions, compare_translit, DIM_FEATURE_NAMES
from src.pair_features import build_matrix, ALL_FEATURE_NAMES

ALL=FEATURE_NAMES+NAME_FEATURE_NAMES+STRING_FEATURE_NAMES+NEIGHBOUR_FEATURE_NAMES+BRAND_FEATURE_NAMES+DIM_FEATURE_NAMES
log(f"порядок столбцов совпадает: {tuple(ALL)==tuple(ALL_FEATURE_NAMES)}")
if tuple(ALL)!=tuple(ALL_FEATURE_NAMES):
    a=[x for x in ALL if x not in set(ALL_FEATURE_NAMES)]; b=[x for x in ALL_FEATURE_NAMES if x not in set(ALL)]
    log(f"  только в обучающем: {a[:8]}"); log(f"  только в инференсном: {b[:8]}")
    diff=[(i,x,y) for i,(x,y) in enumerate(zip(ALL,ALL_FEATURE_NAMES)) if x!=y]
    log(f"  первое расхождение по позиции: {diff[:3]}")

items=pd.read_parquet(base+"/items_human.parquet")
hm=pd.read_parquet(base+"/matches.parquet",columns=["id1","id2","target"]).head(20000).reset_index(drop=True)

# ПУТЬ ОБУЧЕНИЯ: ровно тот код, что считал features_human.npy
def featurize(pool, pairs):
    ID=pool["id"].to_numpy(); NAME=pool["name"].astype(str).tolist(); ATTR=pool["attributes"].tolist()
    CAT=pool["category"].astype(str).to_numpy()
    cards={int(i):parse(n,a,name=n) for i,n,a in zip(ID,NAME,ATTR)}
    nm={int(i):parse_name(n) for i,n in zip(ID,NAME)}
    idf,avg=build_idf(list(nm.values()))
    cols={int(i):colours(n+" "+str(a)) for i,n,a in zip(ID,NAME,ATTR)}
    br={int(i):frozenset(x for x in (canonical(v) for v in k.slots.get("brand",())) if x) for i,k in cards.items()}
    dm={int(i):dimensions(n+" "+str(a)) for i,n,a in zip(ID,NAME,ATTR)}
    POS={int(x):r for r,x in enumerate(ID)}; cat_of=dict(zip(ID.tolist(),CAT.tolist()))
    pc=pairs["id1"].map(cat_of).fillna("?").astype(str).to_numpy()
    out=np.zeros((len(pairs),len(ALL)),dtype=np.float32)
    for cat in sorted(set(CAT)):
        rows=np.flatnonzero(pc==cat)
        if not len(rows): continue
        prof=nb_build(pool[["id","name","category"]],categories=[cat])
        g=pool[pool["category"]==cat]
        M=TfidfVectorizer(min_df=1,sublinear_tf=True).fit_transform(g["name"].astype(str).tolist())
        ix={int(x):r for r,x in enumerate(g["id"].to_numpy())}
        for r in rows:
            a,b=int(pairs["id1"].iat[r]),int(pairs["id2"].iat[r])
            if a not in cards or b not in cards: continue
            s=float((M[ix[a]]@M[ix[b]].T).toarray()[0,0]) if (a in ix and b in ix) else 0.0
            d=compare(cards[a],cards[b]); d.update(compare_names(nm[a],nm[b],idf,avg))
            d.update(compare_strings(NAME[POS[a]],NAME[POS[b]])); d.update(nb_compare(a,b,s,prof))
            d.update(compare_brands(br[a],br[b],{})); d.update(compare_colours(cols[a],cols[b]))
            d.update(compare_dimensions(dm[a],dm[b])); d.update(compare_translit(br[a],br[b]))
            out[r]=[d[k] for k in ALL]
        del prof,M; gc.collect()
    return out

t=time.perf_counter(); A=featurize(items,hm); log(f"путь обучения: {A.shape} за {time.perf_counter()-t:.0f}с")
t=time.perf_counter(); B=build_matrix(items,hm,with_neighbours=True); log(f"путь инференса: {B.shape} за {time.perf_counter()-t:.0f}с")

d=np.abs(A-B); bad=np.flatnonzero(d.max(axis=0)>1e-6)
log(f"столбцов с расхождением: {len(bad)} из {A.shape[1]}")
for i in bad[:15]:
    log(f"  {ALL[i]:<24} max |разн| {d[:,i].max():.4f}  строк с разницей {int((d[:,i]>1e-6).sum()):,}")
log("СОВПАДАЮТ ПОЛНОСТЬЮ" if len(bad)==0 else "ЕСТЬ РАСХОЖДЕНИЯ")
# и сверка с тем, что реально ушло в обучение
Xh=np.load(prev+"/features_human.npy")[:len(hm)]
d2=np.abs(Xh-A); bad2=np.flatnonzero(d2.max(axis=0)>1e-6)
log(f"\nсверка с сохранённой обучающей матрицей: расходится столбцов {len(bad2)}")
for i in bad2[:10]: log(f"  {ALL[i]:<24} max |разн| {d2[:,i].max():.4f}")
log("готово")
